In [1]:
# 06_counterfactual_dice.ipynb
# Generates DiCE counterfactual explanations for the non-diabetic high-risk cohort.
# Actionability constraints are enforced PER INSTANCE PER RUN: waist circumference may be
# reduced by at most 15 cm, BMI and weight by at most 20%; immutable features (age, height,
# education, income, self-rated health) are held fixed. Because the cap is applied to every
# counterfactual in the run, the aggregate "best" counterfactual also respects the 15 cm
# bound (verified in the summary table). Counterfactuals explain the classifier -- which
# features would move an applicant's predicted risk toward the lower-risk region of model
# space -- they are not causal prevention guarantees. Produces representative cases with ALL
# features and a grayscale figure with BOTH sexes. dpi 600, PNG+PDF, no titles/captions.

import warnings, signal
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
import dice_ml
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Per-case timeout: a few extreme high-risk individuals have no feasible counterfactual
# within the actionability caps, which makes the random search loop; we skip such cases.
class _Timeout(Exception):
    pass
def _handler(signum, frame):
    raise _Timeout()
signal.signal(signal.SIGALRM, _handler)
PER_CASE_TIMEOUT = 6
SEED = 42; np.random.seed(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"; MODELS = ROOT / "models"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]
CATEG = ["sm_presnt", "dr_month", "pa_aerobic"]
CONT = [c for c in PRED if c not in CATEG]
IMMUTABLE = ["age", "HE_ht", "edu", "ho_incm", "D_1_1"]
VARY = [c for c in PRED if c not in IMMUTABLE]        # actionable + constrained
WAIST_CAP = 15.0; PCT_CAP = {"HE_BMI": 0.20, "HE_wt": 0.20}
N_SAMPLE = 100; TOTAL_CF = 4

store = {}          # per sex: dict with orig, cf, red, waist_delta, profiles
summary_rows = []; rep_cases = {}
for sx in ["Male", "Female"]:
    full = pd.read_parquet(DATA / f"scored_{sx.lower()}.parquet")
    data_df = full[PRED + ["diabetes"]].copy()
    for c in CATEG:
        data_df[c] = data_df[c].astype(int).astype(str)
    deploy = xgb.XGBClassifier(); deploy.load_model(MODELS / f"xgb_{sx.lower()}_deploy.json")

    class Wrap:
        def __init__(self, mm): self.m = mm
        def predict_proba(self, X):
            Z = X.copy()
            for c in CATEG:
                if Z[c].dtype == object:
                    Z[c] = Z[c].astype(float)
            return self.m.predict_proba(Z[PRED].astype(float))
        def predict(self, X):
            return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    d = dice_ml.Data(dataframe=data_df, continuous_features=CONT, outcome_name="diabetes")
    m = dice_ml.Model(model=Wrap(deploy), backend="sklearn", model_type="classifier")
    exp = dice_ml.Dice(d, m, method="random")

    hr_nd = pd.read_parquet(DATA / f"highrisk_nondiab_{sx.lower()}.parquet")
    samp = hr_nd.sample(min(N_SAMPLE, len(hr_nd)), random_state=SEED).reset_index(drop=True)
    orig_l, cf_l, red_l, wd_l, profs = [], [], [], [], []
    for i in range(len(samp)):
        row = samp.iloc[[i]][PRED].copy()
        for c in CATEG:
            row[c] = row[c].astype(int).astype(str)
        x = samp.iloc[i]
        pr = {"HE_wc": [float(x.HE_wc - WAIST_CAP), float(x.HE_wc)]}
        for v, cap in PCT_CAP.items():
            pr[v] = [float(x[v] * (1 - cap)), float(x[v])]
        try:
            signal.alarm(PER_CASE_TIMEOUT)
            cf = exp.generate_counterfactuals(row, total_CFs=TOTAL_CF, desired_class=0,
                                              features_to_vary=VARY, permitted_range=pr,
                                              random_seed=SEED)
            signal.alarm(0)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            if cfdf is None or len(cfdf) == 0:
                continue
            Xcf = cfdf[PRED].astype(float)
            probs = deploy.predict_proba(Xcf)[:, 1]
            op = float(deploy.predict_proba(samp.iloc[[i]][PRED].astype(float))[:, 1][0])
            j = int(np.argmin(probs)); cp = float(probs[j])
            orig_l.append(op); cf_l.append(cp); red_l.append(op - cp)
            wd_l.append(float(x.HE_wc - Xcf.iloc[j]["HE_wc"]))
            profs.append((x, Xcf.iloc[j], op, cp))
        except (_Timeout, Exception):
            signal.alarm(0)
            continue
    store[sx] = dict(orig=np.array(orig_l), cf=np.array(cf_l), red=np.array(red_l),
                     wd=np.array(wd_l), profs=profs)
    summary_rows.append([sx, len(red_l), round(np.mean(red_l), 3), round(np.std(red_l), 3),
                         round(np.max(wd_l), 1), round(np.mean(wd_l), 1)])
    # representative case = largest risk reduction; ALL features
    k = int(np.argmax(red_l)); xo, xcf, op, cp = profs[k]
    rep = pd.DataFrame({"Variable": PRED, "Original": [round(float(xo[v]), 2) for v in PRED],
                        "Best_CF": [round(float(xcf[v]), 2) for v in PRED]})
    rep.loc[len(rep)] = ["Predicted probability", round(op, 3), round(cp, 3)]
    rep.loc[len(rep)] = ["Risk reduction (pp)", "", round((op - cp) * 100, 1)]
    rep.to_csv(TAB / f"table_representative_cf_{sx.lower()}.csv", index=False)
    rep_cases[sx] = dict(age=int(xo.age), orig=round(op, 3), cf=round(cp, 3),
                         waist_delta=round(float(xo.HE_wc - xcf["HE_wc"]), 1))
    print(f"[{sx}] CFs for {len(red_l)}/{len(samp)} sampled; mean reduction={np.mean(red_l):.3f}; "
          f"max waist reduction={np.max(wd_l):.1f} cm (cap {WAIST_CAP}) -> cap respected={np.max(wd_l) <= WAIST_CAP+1e-6}")

# ---- Figure: Panel A scatter (orig vs best CF), Panel B risk-reduction histogram; BOTH sexes ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
axes[0].plot([0, 1], [0, 1], ls="--", color="0.4", lw=1, label="No change")
for sx, c, mk in [("Male", "#3b3b3b", "o"), ("Female", "#9a9a9a", "s")]:
    axes[0].scatter(store[sx]["orig"], store[sx]["cf"], s=22, facecolor=c, edgecolor="black",
                    linewidth=0.3, alpha=0.75, marker=mk, label=f"{sx} (n={len(store[sx]['orig'])})")
axes[0].set_xlabel("Original risk probability"); axes[0].set_ylabel("Best-CF risk probability")
axes[0].legend(frameon=False, loc="upper left"); axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].text(0.04, 0.92, "(A)", transform=axes[0].transAxes, fontsize=10, fontweight="bold")
for sx, c, h in [("Male", "#3b3b3b", ""), ("Female", "#9a9a9a", "///")]:
    r = store[sx]["red"]
    axes[1].hist(r, bins=18, color=c, alpha=0.7, edgecolor="black", linewidth=0.5, hatch=h,
                 label=f"{sx} (mean={r.mean():.2f})")
    axes[1].axvline(r.mean(), color=c, ls="--", lw=1.2)
axes[1].set_xlabel("Risk reduction (original - best CF)"); axes[1].set_ylabel("Count")
axes[1].legend(frameon=False); axes[1].text(0.04, 0.92, "(B)", transform=axes[1].transAxes,
                                            fontsize=10, fontweight="bold")
savefig(fig, "fig_counterfactual_distribution")

summ = pd.DataFrame(summary_rows, columns=["Sex", "N_cf", "Mean_reduction", "SD_reduction",
                                           "Max_waist_reduction_cm", "Mean_waist_reduction_cm"])
summ.to_csv(TAB / "table_cf_summary.csv", index=False)
print("\nCounterfactual summary (waist cap = 15 cm enforced per run):")
print(summ.to_string(index=False))
print("Representative cases:", rep_cases)
print("Saved: representative CF tables (all features), summary, distribution figure.")


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:05<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.08it/s]

100%|██████████| 1/1 [00:00<00:00,  2.07it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.05it/s]

100%|██████████| 1/1 [00:00<00:00,  3.04it/s]

[Female] CFs for 85/100 sampled; mean reduction=0.063; max waist reduction=14.4 cm (cap 15.0) -> cap respected=True



Counterfactual summary (waist cap = 15 cm enforced per run):
   Sex  N_cf  Mean_reduction  SD_reduction  Max_waist_reduction_cm  Mean_waist_reduction_cm
  Male    84           0.077         0.065                    14.4                      4.6
Female    85           0.063         0.048                    14.4                      2.7
Representative cases: {'Male': {'age': 69, 'orig': 0.529, 'cf': 0.19, 'waist_delta': 14.4}, 'Female': {'age': 67, 'orig': 0.346, 'cf': 0.084, 'waist_delta': 13.5}}
Saved: representative CF tables (all features), summary, distribution figure.
